In [1]:
# Import necessary libraries
import os
import time
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re
from tqdm import tqdm
import sys

# Initialize tqdm for pandas
tqdm.pandas()

# Debug: Confirm that imports are successful
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module
print(f"Python version: {sys.version}")
print(f"os version: Part of Python standard library, version {sys.version}")
print(f"requests version: {requests.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")

# Set the base directory for datasets in Kaggle
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets
sub_dir = r"/kaggle/working/"  # Submission directory for output files

# Debug: Print the directory paths to confirm they are set correctly
print(f"Debug: Base directory: {base_dir}")
print(f"Debug: Submission directory: {sub_dir}")

# Debug: List the files in the base directory to verify the presence of input files
print(f"Debug: Listing files in {base_dir}:")
try:
    files_in_dir = os.listdir(base_dir)
    print(files_in_dir)
except Exception as e:
    print(f"Error: Could not list files in {base_dir}. Error: {str(e)}")
    raise Exception(f"Failed to access directory {base_dir}")

# File paths for input datasets
EXCEL_PATH = os.path.join(base_dir, "Airquality_Unique_geocode_with_LatLong.xlsx")
POLLUTION_FILE = os.path.join(base_dir, "Air_Quality_20250221.csv")
TRAIN_FILE = os.path.join(base_dir, "Training_data.csv")
VALIDATION_FILE = os.path.join(base_dir, "Validation_data.csv")

# Intermediate output for geocoded data
GEOCODES_OUTPUT = os.path.join(sub_dir, "Airquality_Unique_geocode_with_LatLong.xlsx")

# Final output CSVs
TRAIN_OUTPUT = os.path.join(sub_dir, "training_data_AIRPOLLUTION.csv")
VALIDATION_OUTPUT = os.path.join(sub_dir, "validation_data_AIRPOLLUTION.csv")

# Debug: Check if input files exist
print(f"Debug: Air quality geocode file exists: {os.path.exists(EXCEL_PATH)}")
print(f"Debug: Pollution data file exists: {os.path.exists(POLLUTION_FILE)}")
print(f"Debug: Training dataset file exists: {os.path.exists(TRAIN_FILE)}")
print(f"Debug: Validation dataset file exists: {os.path.exists(VALIDATION_FILE)}")

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
os version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
requests version: 2.32.3
pandas version: 2.2.3
geopandas version: 0.14.4
Debug: Base directory: /kaggle/input/eyds-base-dataset
Debug: Submission directory: /kaggle/working/
Debug: Listing files in /kaggle/input/eyds-base-dataset:
['census_block_loc.csv', 'Hyperlocal_Temperature_Monitoring_20250311.csv', 'Airquality_Unique_geocode_with_LatLong.xlsx', 'nyclion_25a', 'StreetAssessmentRating', 'Sentinel2', 'USA_wind-speed_10m.tif', 'USA_power-density_10m.tif', 'Validation_data.csv', 'LSAT_8_221022', 'Training_data.csv', 'NYC_Cooling_Tower_Registrations_20250224.csv', 'AQ', 'S2_DATA.tiff', 'Automated_Traffic_Volume_Counts_20250319.csv', 'Air_Quality_20250221.csv', 'USA_air-density_10m.tif', 'nclimgrid-monthly-202107.tif', 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data

In [2]:
#############################################
# Part A: Geocode Air Quality Data Using TomTom API
#############################################

# 1. Parameters for TomTom API
TOMTOM_API_KEY = "#"  # Replace with your actual API key
BASE_URL = "https://api.tomtom.com/search/2/geocode"

# 2. Read the Excel file with place names
print("Loading air quality geocode data...")
try:
    geocodes_df = pd.read_excel(EXCEL_PATH)
except FileNotFoundError as e:
    print(f"Error: Air quality geocode file not found at {EXCEL_PATH}. Error: {str(e)}")
    raise Exception("Failed to load air quality geocode dataset")

print("Air quality geocode data loaded with", geocodes_df.shape[0], "rows.")
print("Debug: Air quality geocode columns:", geocodes_df.columns.tolist())

# 3. Create columns for Latitude and Longitude if they don't exist
if 'Latitude' not in geocodes_df.columns:
    geocodes_df['Latitude'] = 0.0
if 'Longitude' not in geocodes_df.columns:
    geocodes_df['Longitude'] = 0.0

# 4. Function to query TomTom's Geocoding API with bounding around NYC
def get_lat_lon_from_tomtom(place):
    """
    Given a place name, returns (latitude, longitude) from TomTom's Geocoding API,
    restricted to the US and within ~50 km of Manhattan's center.
    Returns (0, 0) if no match is found or if there's an error.
    """
    try:
        # Encode the place to safely include it in the URL
        place_encoded = requests.utils.quote(place)
        url = f"{BASE_URL}/{place_encoded}.json"
        params = {
            "key": TOMTOM_API_KEY,
            "limit": 1,
            "countrySet": "US",       # Restrict to United States
            "lat": 40.7128,          # Approx. center of Manhattan
            "lon": -74.0060,         # Approx. center of Manhattan
            "radius": 50000,         # 50 km around Manhattan
            "language": "en-US"      # Optional: request English results
        }
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        results = data.get("results", [])
        if results:
            position = results[0].get("position", {})
            lat = position.get("lat", 0)
            lon = position.get("lon", 0)
            return (lat, lon)
        else:
            return (0, 0)
    except Exception as e:
        print(f"Debug: Error geocoding place '{place}': {str(e)}")
        return (0, 0)

# 5. Loop over each row and geocode with progress tracking
print("Geocoding air quality place names...")
for idx, row in tqdm(geocodes_df.iterrows(), total=len(geocodes_df), desc="Geocoding"):
    place = row.get('Cleaned_Geo_Place_Name', '')
    if not place or pd.isna(place):
        # If place is missing or NaN, skip
        geocodes_df.at[idx, 'Latitude'] = 0
        geocodes_df.at[idx, 'Longitude'] = 0
        continue

    # For better accuracy, append ", New York" to the place name
    place_query = place + ", New York"

    lat, lon = get_lat_lon_from_tomtom(place_query)
    geocodes_df.at[idx, 'Latitude'] = lat
    geocodes_df.at[idx, 'Longitude'] = lon

    # Respect rate limits / avoid hitting the API too fast
    time.sleep(0.1)  # Adjust as needed based on API rate limits

# 6. Save the geocoded DataFrame to a new Excel file
geocodes_df.to_excel(GEOCODES_OUTPUT, index=False)
print(f"Geocoding complete. Results saved to: {GEOCODES_OUTPUT}")

# Debug: Check for missing or invalid coordinates
missing_coords = geocodes_df[(geocodes_df['Latitude'] == 0) & (geocodes_df['Longitude'] == 0)].shape[0]
print(f"Debug: Number of rows with missing/invalid coordinates: {missing_coords}")

Loading air quality geocode data...
Air quality geocode data loaded with 109 rows.
Debug: Air quality geocode columns: ['Cleaned_Geo_Place_Name', 'Latitude', 'Longitude']
Geocoding air quality place names...


Geocoding:   1%|          | 1/109 [00:00<00:16,  6.49it/s]

Debug: Error geocoding place 'Southeast Queens, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Southeast%20Queens%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US
Debug: Error geocoding place 'Upper East Side-Gramercy, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Upper%20East%20Side-Gramercy%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US


Geocoding:   2%|▏         | 2/109 [00:00<00:15,  6.72it/s]

Debug: Error geocoding place 'Fordham - Bronx Pk, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Fordham%20-%20Bronx%20Pk%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US


...

Debug: Error geocoding place 'Morningside Heights and Hamilton Heights, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Morningside%20Heights%20and%20Hamilton%20Heights%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US
Debug: Error geocoding place 'Bensonhurst, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Bensonhurst%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US


Geocoding:  98%|█████████▊| 107/109 [00:15<00:00,  7.05it/s]

Debug: Error geocoding place 'Sheepshead Bay, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Sheepshead%20Bay%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US
Debug: Error geocoding place 'Ridgewood and Maspeth, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Ridgewood%20and%20Maspeth%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US


Geocoding: 100%|██████████| 109/109 [00:15<00:00,  6.86it/s]

Debug: Error geocoding place 'Willowbrook, New York': 403 Client Error: Forbidden for url: https://api.tomtom.com/search/2/geocode/Willowbrook%2C%20New%20York.json?key=%23&limit=1&countrySet=US&lat=40.7128&lon=-74.006&radius=50000&language=en-US
Geocoding complete. Results saved to: /kaggle/working/Airquality_Unique_geocode_with_LatLong.xlsx
Debug: Number of rows with missing/invalid coordinates: 109


In [3]:
#############################################
# Part B: Link Geocoded Data with Pollution Values
#############################################

# 7. Load the pollution data
print("Loading pollution data...")
try:
    pollution_df = pd.read_csv(POLLUTION_FILE)
except FileNotFoundError as e:
    print(f"Error: Pollution data file not found at {POLLUTION_FILE}. Error: {str(e)}")
    raise Exception("Failed to load pollution dataset")

print("Pollution data loaded with", pollution_df.shape[0], "rows.")
print("Debug: Pollution data columns:", pollution_df.columns.tolist())

# 8. Filter & Pivot the Pollution Data
# Define the time period filters for each pollutant category
ozone_filter = (pollution_df['Name'] == "Ozone (O3)") & (pollution_df['Time Period'].str.contains("Summer"))
fine_no2_filter = (pollution_df['Name'].isin(["Fine particles (PM 2.5)", "Nitrogen dioxide (NO2)"])) & \
                  (pollution_df['Time Period'].str.contains("Annual Average") | pollution_df['Time Period'].str.contains("Summer") | pollution_df['Time Period'].str.contains("Winter"))

# Combine filters
pollution_filtered = pollution_df[ozone_filter | fine_no2_filter].copy()

# Create a new column that combines pollutant name and time period (e.g., "Ozone (O3) Summer")
pollution_filtered['Pollutant_Time'] = pollution_filtered['Name'] + " " + pollution_filtered['Time Period'].str.extract(r'(\bAnnual Average\b|\bSummer\b|\bWinter\b)', expand=False)

# Pivot so each "Geo Place Name" has columns for each pollutant-time period combination
pollution_pivot = pollution_filtered.pivot_table(
    index='Geo Place Name',
    columns='Pollutant_Time',
    values='Data Value'
).reset_index()

# Clean up "Geo Place Name" by removing any parentheses and extra whitespace
pollution_pivot['Geo Place Name'] = pollution_pivot['Geo Place Name'].str.replace(r'\(.*?\)', '', regex=True).str.strip()

# Debug: Check the pivoted pollution data
print("Debug: Pivoted pollution data shape:", pollution_pivot.shape)
print("Debug: Pivoted pollution data columns:", pollution_pivot.columns.tolist())
print("Debug: Sample pivoted pollution data (first 5 rows):")
print(pollution_pivot.head())

# 9. Merge Pivoted Pollution with Geocodes
# Rename columns to match if needed
geocodes_df.rename(columns={"Cleaned_Geo_Place_Name": "Geo Place Name"}, inplace=True)

# Merge pivoted data with geocode lat/long data
pollution_merged = pd.merge(
    pollution_pivot, 
    geocodes_df[["Geo Place Name", "Latitude", "Longitude"]],
    on="Geo Place Name",
    how="left"
)

# Drop rows with invalid coordinates
pollution_merged.dropna(subset=['Latitude', 'Longitude'], inplace=True)
pollution_merged = pollution_merged[(pollution_merged['Latitude'] != 0) & (pollution_merged['Longitude'] != 0)]

# Drop duplicate locations if any exist (so each unique lat/long appears only once)
pollution_merged = pollution_merged.drop_duplicates(subset=['Latitude', 'Longitude'])

# Debug: Check the merged pollution data
print("Debug: Merged pollution data shape:", pollution_merged.shape)
print("Debug: Merged pollution data columns:", pollution_merged.columns.tolist())
print("Debug: Sample merged pollution data (first 5 rows):")
print(pollution_merged.head())

# 10. Create a GeoDataFrame for Pollution
pollution_gdf = gpd.GeoDataFrame(
    pollution_merged,
    geometry=gpd.points_from_xy(pollution_merged['Longitude'], pollution_merged['Latitude']),
    crs="EPSG:4326"
)

Loading pollution data...
Pollution data loaded with 18025 rows.
Debug: Pollution data columns: ['Unique ID', 'Indicator ID', 'Name', 'Measure', 'Measure Info', 'Geo Type Name', 'Geo Join ID', 'Geo Place Name', 'Time Period', 'Start_Date', 'Data Value', 'Message']
Debug: Pivoted pollution data shape: (114, 8)
Debug: Pivoted pollution data columns: ['Geo Place Name', 'Fine particles (PM 2.5) Annual Average', 'Fine particles (PM 2.5) Summer', 'Fine particles (PM 2.5) Winter', 'Nitrogen dioxide (NO2) Annual Average', 'Nitrogen dioxide (NO2) Summer', 'Nitrogen dioxide (NO2) Winter', 'Ozone (O3) Summer']
Debug: Sample pivoted pollution data (first 5 rows):
Pollutant_Time                     Geo Place Name  \
0                     Bay Ridge and Dyker Heights   
1                           Bayside - Little Neck   
2               Bayside Little Neck-Fresh Meadows   
3                         Bayside and Little Neck   
4                              Bedford Stuyvesant   

Pollutant_Time  Fine 

In [4]:
#############################################
# Part C: Join Pollution Data with Training and Validation Sets
#############################################

# 11. Load the training and validation datasets and convert them to GeoDataFrames
print("Loading training data...")
try:
    train_df = pd.read_csv(TRAIN_FILE)
except FileNotFoundError:
    print(f"Error: Training data file not found at {TRAIN_FILE}")
    raise Exception("Failed to load training dataset")

print("Training data loaded with", train_df.shape[0], "rows.")
train_gdf = gpd.GeoDataFrame(
    train_df.copy().reset_index(),  # original index stored in "index"
    geometry=gpd.points_from_xy(train_df.Longitude, train_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Training GeoDataFrame shape: {train_gdf.shape}")
print(f"Debug: Training GeoDataFrame columns: {train_gdf.columns.tolist()}")

print("Loading validation data...")
try:
    val_df = pd.read_csv(VALIDATION_FILE)
except FileNotFoundError:
    print(f"Error: Validation data file not found at {VALIDATION_FILE}")
    raise Exception("Failed to load validation dataset")

print("Validation data loaded with", val_df.shape[0], "rows.")
val_gdf = gpd.GeoDataFrame(
    val_df.copy().reset_index(),
    geometry=gpd.points_from_xy(val_df.Longitude, val_df.Latitude),
    crs="EPSG:4326"
)
print(f"Debug: Validation GeoDataFrame shape: {val_gdf.shape}")
print(f"Debug: Validation GeoDataFrame columns: {val_gdf.columns.tolist()}")

# 12. Nearest-Neighbor Join with Training & Validation
# Perform a spatial nearest join. This may return duplicate rows if multiple pollution points are equidistant.
train_joined = gpd.sjoin_nearest(
    train_gdf,
    pollution_gdf,
    how='left',
    distance_col='dist_to_pollution'
)

val_joined = gpd.sjoin_nearest(
    val_gdf,
    pollution_gdf,
    how='left',
    distance_col='dist_to_pollution'
)

# To ensure one output per input row, group by the original row identifier and keep the closest match
train_joined = train_joined.sort_values('dist_to_pollution').drop_duplicates(subset='index', keep='first')
val_joined = val_joined.sort_values('dist_to_pollution').drop_duplicates(subset='index', keep='first')

# Reset the index to match original data
train_joined = train_joined.set_index('index')
val_joined = val_joined.set_index('index')

# Debug: Check the joined data
print("Debug: Joined training data shape:", train_joined.shape)
print("Debug: Joined training data columns:", train_joined.columns.tolist())
print("Debug: Sample joined training data (first 5 rows):")
print(train_joined.head())

print("Debug: Joined validation data shape:", val_joined.shape)
print("Debug: Joined validation data columns:", val_joined.columns.tolist())
print("Debug: Sample joined validation data (first 5 rows):")
print(val_joined.head())

Loading training data...
Training data loaded with 11229 rows.
Debug: Training GeoDataFrame shape: (11229, 6)
Debug: Training GeoDataFrame columns: ['index', 'Longitude', 'Latitude', 'datetime', 'UHI Index', 'geometry']
Loading validation data...
Validation data loaded with 1040 rows.
Debug: Validation GeoDataFrame shape: (1040, 5)
Debug: Validation GeoDataFrame columns: ['index', 'Longitude', 'Latitude', 'UHI Index', 'geometry']
Debug: Joined training data shape: (11229, 17)
Debug: Joined training data columns: ['Longitude_left', 'Latitude_left', 'datetime', 'UHI Index', 'geometry', 'index_right', 'Geo Place Name', 'Fine particles (PM 2.5) Annual Average', 'Fine particles (PM 2.5) Summer', 'Fine particles (PM 2.5) Winter', 'Nitrogen dioxide (NO2) Annual Average', 'Nitrogen dioxide (NO2) Summer', 'Nitrogen dioxide (NO2) Winter', 'Ozone (O3) Summer', 'Latitude_right', 'Longitude_right', 'dist_to_pollution']
Debug: Sample joined training data (first 5 rows):
       Longitude_left  Latitu

/usr/local/lib/python3.10/dist-packages/geopandas/array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/geopandas/array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWa

In [5]:
#############################################
# Part D: Clean Up & Save Final CSVs
#############################################

# 13. Clean up the final DataFrames
cols_to_drop = ['geometry', 'index_right', 'dist_to_pollution', 'Geo Place Name', 'datetime', 'UHI Index']
train_final = train_joined.drop(columns=cols_to_drop, errors='ignore')
val_final = val_joined.drop(columns=cols_to_drop, errors='ignore')

# 14. Save the final DataFrames to CSV
train_final.to_csv(TRAIN_OUTPUT, index=False)
val_final.to_csv(VALIDATION_OUTPUT, index=False)

# Debug: Print the final confirmation messages with file paths
print(f"Debug: Training data with pollution features saved to: {TRAIN_OUTPUT}")
print(f"Debug: Validation data with pollution features saved to: {VALIDATION_OUTPUT}")
print("Done! Pollution columns added to both training and validation datasets.")
print(f"Train output: {TRAIN_OUTPUT}")
print(f"Validation output: {VALIDATION_OUTPUT}")

Debug: Training data with pollution features saved to: /kaggle/working/training_data_AIRPOLLUTION.csv
Debug: Validation data with pollution features saved to: /kaggle/working/validation_data_AIRPOLLUTION.csv
Done! Pollution columns added to both training and validation datasets.
Train output: /kaggle/working/training_data_AIRPOLLUTION.csv
Validation output: /kaggle/working/validation_data_AIRPOLLUTION.csv
